In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

# Decision Trees

Применяется в задачах регрессии, классификации и ранжирования

## Работа алгоритма
### - fit -

Создаётся корень, выбирается ```правило```, по которому данные будут делиться на два подмножества ```left``` и ```right``` nodes\
\
Узел перестанет делиться (Находить ```правило```, по которому будет делить данные на два множества), если:
- Узел станет ```чистым``` - если в узле все объекты имеют одно и то же значение целевой переменной, то дальнейшее деление бессмысленно
- Будет достигнут лимит глубины дерева
- Количество объектов в узле меньше заданного порога


После того, как узел становится листовым, ему присваивают значение целевой переменной, которое выбирается:\
**методом большинства** |\
**методом большинства, но с добавлением каждому классу собственного веса** |\
**расчетом вероятности принадлежности к классу** |\
**средне-арифметическим значений целевой переменной** - в общем, в лист можно записать в качестве значения целевой метрики много всего, по большей мере выбор зависит от тонкостей конкретной задачи


```правила```:
- ```Индекс Джини```:\
  Сначала рассчитаем формулу для узла
  $$P(ошибки) = 1 - P(успеха)$$
  предположим, что мы "случайно ткнули и угадали", чтобы такое произошло, мы должны выбрать i_тое значение целевой переменной и ткнув в случайный элемент "угадать его", то есть чтобы он оказался i_того значения целевой переменной, пусть p_i = кол-во_i_тых_значений_целевой / кол-во_объектов_в_узле, тогда вероятность нашего события $$p_i \cdot p_i = p_i^2$$
  Теперь найдем вероятность того, что мы "угадали класс", то есть нам нужно одновременное выполнение условий "выбран i_тый класс" и "попался i_тый класс", потому $$ P(угадывания \ класса) = \sum_{i=1}^Cp_i^2, \ где \ \ C - \ количество \ классов$$ Теперь ищем ошибку, то есть
  $$ P(Ошибки) = 1 - P(угадывания \ класса)$$
  $$ P(Ошибки) = 1 - \sum_{i=1}^Cp_i^2 $$
  Получили формулу Джини
  $$ Gini = 1 - \sum_{i=1}^Cp_i^2 $$
  Теперь с помощью этой формулы будем оценивать, насколько хорошо то или иное разбиение, предположим, что некоторым ```алгоритмом подборов вариантов разбиений``` мы получили k разбиений на left и right узлы, посчитаем для каждого критерий Gini
  $$Gini_{left} = 1 - \sum_{i=1}^{C\_left}p\_left_i^2$$
  $$Gini_{right} = 1 - \sum_{i=1}^{C\_right}p\_right^2$$
  Вероятность того, что случайный объект принадлежит левому
  $$P(left) = \frac{N_{left}}{N}, \ где \ N_{left} \ - \ количество \ элементов \ в \ \text{left} \ узле$$
  Вероятность того, что случайный объект принадлежит правому
  $$P(right) = \frac{N_{right}}{N}, \ где \ N_{right} \ - \ количество \ элементов \ в \ \text{right} \ узле$$
  Теперь посчитаем математическое ожидание для Gini_j_разбиения
  $$ Gini_{split_j} = P(left) \cdot Gini_{left} + P(right) \cdot Gini_{right}$$
  В результате получаем формулу для оценки j-того разбиения 
  $$ Gini_{split_j} = \frac{N_{left}}{N} \cdot Gini_{left} + \frac{N_{right}}{N} \cdot Gini_{right}$$

  В итоге выбирается разбиение с наименьшим индексом Джини (min(Gini_split_j))
- ```Критерий энтропии```
  $$H = -\sum_{i=1}^{C} p_i \cdot log_2 (p_i)$$
  предположим, что некоторым ```алгоритмом подборов вариантов разбиений``` мы получили k разбиений на left и right узлы, посчитаем для каждого энтропию по формуле Шеннона

  Считаем для левого узла
  $$H_{left} = -\sum_{i=1}^{C} p\_left_i \cdot log_2 (p\_left_i)$$
  Считаем для правого узла
  $$H_{right} = -\sum_{i=1}^{C} p\_right_i \cdot log_2 (p\_right_i)$$

  Считаем взвешенную энтропию
  $$H_{split\_j} = \frac{N_{left}}{N} \cdot H_{left} + \frac{N_{right}}{N} \cdot H_{right}$$

  В итоге выбирается разбиение с наименьшей энтропией

  Также хочется заметить, что иногда считают информационный выигрыш
  $$ H_{parent} - H_{split_j} $$
  он нам показывает, насколько этот узел понизит энтропию относительно родительского узла\
  С его помощью также можно строить правила выбора разбиения или, например, добавлять логику вида "Если максимальный информационный выигрыш меньше 0.001, то сделай узел листовым"


Рассмотрев данные правила, можно заметить, что они идут по одному сценарию
1) некоторым ```алгоритмом подборов вариантов разбиений``` находятся k разбиений
2) для каждого left и right разбиений считается некоторый ```критерий```
3) считаем математическое ожидание для каждого разбиения
4) по некоторой оценке критерия выбираем лучшее разбиение\
Например, по тому же принципу рассчитывается MSE для случаев, когда деревья используются в задачах регрессии. По формуле для узла $$\sigma = \frac{1}{N} \sum_{i=1}^{N} (y_i - \bar{y})^2, \ где \ \bar{y} \ - \ \text{ср-ариф} \ значений \ у \ в \ узле $$\

### -----------------------------------------------------
### ```алгоритмы подборов вариантов разбиений```

| Тип признака | Алгоритм |
| :-: | :-: |
| Числовой | Берутся все уникальные значения, сортируются, а после считаются $$порог_i = \frac{x_i + x_{i+1}}{2}$$ по итогу выходит $$кол-во \ уникальных \ значений \ - \ 1$$ порогов, по которым разбивают узел, в более ресурсоемких моделях прибегают к группировке данных в бины (Как в гистограммах) и порогом делают пограничные у бинов значения|
| Категориальный | Считается среднее значение целевой переменной, доля положительного класса (В зависимости от типа целевой переменной), после чего категориальные значения сортируются по посчитанному значению и делятся между собой как в числовом варианте (Target encoding подходит лишь для регрессии и бинарной классификации) | 

  
### - inference -

Начинаем с корня, далее по правилу в узле делаем выбор идти в left или right и так, пока не дойдем до листового узла, в нем и будет храниться ответ модели



# Decision Trees (Laboratory)
## 4.7         (Decision Trees)  In this assignment you will train a classification tree, using the survey data from Lab Exercise 7: Logistic Regression.  Import the survey data survey.csv from Lab Exercise 7: Logistic Regression into R, storing rows 1 through 600 as training data and rows 601 through 750 as testing data.



In [ ]:
raw = pd.read_csv("resources/5-lab/survey.csv")
raw.head(2)

In [ ]:
train_df = raw.iloc[:600]
test_df = raw.iloc[600:]

Y_train = train_df['MYDEPV'].astype(np.int8)
X_train = train_df.drop(columns=['MYDEPV']).astype(np.int8)

Y_test = test_df['MYDEPV'].astype(np.int8)
X_test = test_df.drop(columns=['MYDEPV']).astype(np.int8)

### a.      Build a classification tree from the training data using the “rpart” package, according to the formula “MYDEPV ~ Price + Income + Age”.  Use three-fold cross-validation and the information gain splitting index.  Which features were actually used to construct the tree?  (see the “printcp” function)  Plot the tree using the “rpart.plot” package.


In [ ]:
dtree = DecisionTreeClassifier(criterion='entropy', random_state=42)

Модель хорошо предсказывает целевую переменную при выбранных признаках (Угадывание в 92% случаев в среднем)

In [ ]:
cross_val_score(dtree, X_train, Y_train, cv=3)

использовались все 3 признака
Price, Income, Age\
Узлы выводятся в порядке прямого обхода в глубину\
```-2``` является значением листового узла

In [ ]:
dtree.fit(X_train, Y_train).tree_.feature

In [ ]:
plt.figure(figsize=(52, 26))

plot_tree(
    dtree, 
    feature_names = ['Price', 'Income', 'Age'], 
    class_names = ['0', '1'], 
    filled = True, 
    rounded = True,
    fontsize = 9 
)

plt.savefig('resources/information_gain.svg', format='svg', bbox_inches='tight')

plt.show()

### b.      Score the model with the training data and create the model’s confusion matrix.  Which class of MYDEPV was the model better able to classify?

Модель хорошо справляется с классификацией 0 класса, с 1 классом также хороший показатель, но несколько ошибок случилось\
В следствие можно выдвинуть гипотезу, что в листах с ненулевой энтропией предпочтение отдается нулевому классу

In [ ]:
Y_tr = dtree.predict(X_train)
cm = confusion_matrix(Y_train, Y_tr)

pd.DataFrame(cm)

### c.       Define the resubstitution error rate, and then calculate it using the confusion matrix from the previous step.  Is it a good indicator of predictive performance?  Why or why not?
Ошибка в 1% очень хороший показатель для модели, но мы не можем, основываясь на этих данных, судить о качестве модели, тк мы проверяем ее на тренировочных данных, в следствие чего имеет место быть оптимистическая смещенность - модель обучена на этих данных и подстроена к ним
Из этого эксперимента можно заключить, что модель хорошо подстроилась под наши тренировочные данные, но нельзя ничего утверждать по поводу её обобщающих способностей и возможности правильно предсказать новые объекты

In [ ]:
print(f"Resubstitution error rate is {(cm[0][1] + cm[1][0]) / np.array(cm).sum()}")

### d.      Using the “ROCR” package, plot the receiver operating characteristic (ROC) curve.  Note: For a refresher on how to use the ROCR package, see Lab Exercise 7: Logistic Regression.  Calculate the area under the ROC curve (AUC).  Describe the usefulness of this statistic.

In [ ]:
Y_probs = dtree.predict_proba(X_train)[:, 1]


# FPR - False Positive Rate (Доля ложноположительных результатов)
# TPR - True Positive Rate (Доля истинно положительных результатов)
# thresholds - пороги
FPR, TPR, thresholds = roc_curve(Y_train, Y_probs)

# AUC - Area Under Curve (Площадь под кривой)
AUC = roc_auc_score(Y_train, Y_probs)

plt.figure(figsize=(4, 4))
plt.plot(FPR, TPR, color='darkorange', lw=2, label=f'ROC')
plt.plot([0, 1], [0, 1], color='black', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC')
plt.grid()

plt.show()

Можно заметить, что модель дошла до у=1.0 быстро, это означает, что при дальнейшем понижении порога она выдает уже ложноположительные значения

### e.      Score the model with the testing data.  How accurate are the tree’s predictions?

In [ ]:
Y_pred = dtree.predict(X_test)
display(pd.DataFrame(confusion_matrix(Y_test, Y_pred)))

In [ ]:
Y_probs = dtree.predict_proba(X_test)[:, 1]

# FPR - False Positive Rate (Доля ложноположительных результатов)
# TPR - True Positive Rate (Доля истинно положительных результатов)
# thresholds - пороги
FPR, TPR, thresholds = roc_curve(Y_test, Y_probs)

# AUC - Area Under Curve (Площадь под кривой)
AUC = roc_auc_score(Y_test, Y_probs)

plt.figure(figsize=(4, 4))
plt.plot(FPR, TPR, color='darkorange', lw=2, label=f'ROC')
plt.plot([0, 1], [0, 1], color='black', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC')
plt.grid()

plt.show()

Хотя результаты чуть хуже предыдущих, модель всё еще показывает хороший результат

### f.        Repeat part (a), but set the splitting index to the Gini coefficient splitting index.  How does the new tree compare to the previous one? 

In [ ]:
dtree = DecisionTreeClassifier(criterion='gini', random_state=42)

In [ ]:
cross_val_score(dtree, X_train, Y_train, cv=3)

In [ ]:
dtree.fit(X_train, Y_train).tree_.feature

In [ ]:
plt.figure(figsize=(52, 26))

plot_tree(
    dtree, 
    feature_names = ['Price', 'Income', 'Age'], 
    class_names = ['0', '1'], 
    filled = True, 
    rounded = True,
    fontsize = 9 
)

plt.savefig('resources/gini.svg', format='svg', bbox_inches='tight')

plt.show()

In [ ]:
Y_tr = dtree.predict(X_train)
cm = confusion_matrix(Y_train, Y_tr)

display(pd.DataFrame(cm))

Y_tr = dtree.predict(X_test)
cm = confusion_matrix(Y_test, Y_tr)

for_future_comparing = pd.DataFrame(cm)
for_future_comparing


| Дерево "информационный выигрыш" | Дерево "Критерий Джини" |
| :-: | :-: |
|  ![](resources/information_gain.svg)|![](resources/gini.svg) |


Деревья вышли похожими, как по графику, так и по матрицам ошибок


Это произошло, потому что критерий Джини и энтропия похожи природой:\
Критерий Джини считает вероятность ошибиться\
Энтропия - мера хаоса, из которой тоже следует "оценка вероятности попасть не туда"

В следствие чего по этим двум правилам часто получаются похожие или идентичные деревья

### g.      Pruning is a technique that reduces the size/depth of a decision tree by removing sections with low classification power, which helps reduce overfitting and simplifies the model, reducing the computational cost.  One way to prune a tree is according to the complexity parameter associated with the smallest cross-validation error.  Prune the new tree in this way using the “prune” function.  Which features were actually used in the pruned tree?  Why were certain variables not used?

Прунинг - обрезка дерева путем удаления "неудовлетворяющих" ветвей, помогает избавиться от переобучения

| название | алгоритм | 
| :-: | :-: |
| Cost-Complexity Pruning (CCP) | Вводится понятие стоимости дерева $$Стоимость\_дерева = Ошибка + \alpha \cdot (Количество\_листьев),$$ $$где \ \alpha \ - \ некоторый \ коэффициент \ (Штраф \ за \ сложность)$$  Алгоритм находт и удаляет те ветки, чье удаление наименьше увеличивает ошибку| 
| Reduced Error Pruning (REP) | Для данного алгоритма нужна дополнительная отдельная выборка данных, алгоритм снизу-вверх пробует каждый узел заменить на лист и если точность модели не ухудшилась, ветка удаляется |
| Pessimistic Error Pruning (PEP) | Работает как REP, но использует не отдельную выборку данных, а делает на тренировочных данных, поскольку ошибка на train данных занижена, накладывает листу дополнительный штраф (Для компенсации переобучения)|
| Minimum Error Pruning (MEP) | Байесовский подход. Оценивается вероятность правильной классификации в каждом узле, если ожидаемая ошибка для поддерева оказывается выше, чем без него, то поддерево удаляется | 

In [ ]:
# находим пути-обрезки 
p = dtree.cost_complexity_pruning_path(X_train, Y_train)

cur_alpha = p.ccp_alphas[0]
ma_score = -1

for a in p.ccp_alphas:
    tree = DecisionTreeClassifier(criterion='gini', ccp_alpha=a, random_state=42)
    
    score = cross_val_score(tree, X_train, Y_train, cv=3, scoring='accuracy').mean()

    if score > ma_score:
        ma_score = score
        cur_alpha = a

print(f"The best ccp_alpa is {cur_alpha}")

In [ ]:
dtree = DecisionTreeClassifier(criterion='gini', ccp_alpha=cur_alpha, random_state=42)
dtree.fit(X_train, Y_train)

Нет ни одного нулевого по значимости признака =>\
Price Income Age использовались

In [ ]:
dtree.feature_importances_

### h.      Create the confusion matrix for the new model, and compare the performance of the model before and after pruning.

Матрицы ошибок почти не различаются, но всё же на один объект классификация стала лучше
При обработке индустриальных объёмов данных, прирост точности модели в 0.006 мог бы принести неплохую выгоду, в связи с чем прунинг оказался полезным для данной модели

In [ ]:
Y_tr = dtree.predict(X_test)
cm = confusion_matrix(Y_test, Y_tr)

display(pd.DataFrame(cm)) # новое дерево
for_future_comparing # старое дерево

Результирующее дерево после прунинга:

In [ ]:
plt.figure(figsize=(52, 26))

plot_tree(
    dtree, 
    feature_names = ['Price', 'Income', 'Age'], 
    class_names = ['0', '1'], 
    filled = True, 
    rounded = True,
    fontsize = 9 
)

plt.show()